# 8. Building Interactive Applications with Databricks Apps

Databricks Apps lets you build and deploy production-quality web applications directly from your data platform. Turn your data insights, ML models, and AI agents into interactive applications that your team or customers can use—without managing separate infrastructure.

## What are Databricks Apps?

**Databricks Apps** enables you to deploy interactive web applications using popular Python frameworks like Streamlit, Gradio, Dash, and Flask. These apps run natively on Databricks, with built-in access to your data, models, and security.

### Common Use Cases:
* Internal tools and dashboards for business teams
* ML model inference interfaces
* AI agent chatbots and assistants
* Data exploration and analysis tools
* Customer-facing data applications

### Key Benefits:
* No separate infrastructure to manage
* Automatic Unity Catalog integration
* Built-in authentication and governance
* Scale from prototype to production seamlessly

## Databricks Apps Overview (Video)
[![Video Thumbnail](https://img.youtube.com/vi/Equ7PBeM-Mw/0.jpg)](https://www.youtube.com/watch?v=Equ7PBeM-Mw)

📖 **Resource:** [Databricks Apps Documentation](https://docs.databricks.com/en/dev-tools/databricks-apps/index.html)

## Method 1: Creating Apps from the UI

The quickest way to deploy an app is through the Databricks workspace UI.

### To Create Your First App:
1. In the left navigation bar, click **+ New** > **App**
2. Choose your framework (Streamlit, Gradio, Dash, or Flask)
3. Select a compute resource or create a new one
4. Write your app code in the built-in editor or link to a repository
5. Click **Deploy** to publish your app

Your app will be accessible via a secure URL that you can share with your team.

📖 **Resource:** [Create and deploy apps](https://docs.databricks.com/en/dev-tools/databricks-apps/create-app.html)

## Method 2: Building Apps with Code (Streamlit)

For developers who prefer working in notebooks or IDEs, you can build and deploy apps programmatically. **Streamlit** is the most popular framework for data apps due to its simplicity.

Below is a minimal example that queries a Unity Catalog table and displays interactive visualizations.

In [ ]:
# Example: Simple Streamlit app that queries data and displays it
# Save this as app.py in your workspace or repository

import streamlit as st
from databricks import sql
import pandas as pd
import os

st.title("📊 Sales Analytics Dashboard")
st.write("Interactive data exploration powered by Databricks")

# Connect to SQL Warehouse
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    
    with connection.cursor() as cursor:
        # Query Unity Catalog table
        cursor.execute("SELECT * FROM main.default.sales_data LIMIT 1000")
        df = cursor.fetchall_arrow().to_pandas()

# Display interactive filters
region = st.selectbox("Select Region", df['region'].unique())
filtered_df = df[df['region'] == region]

# Show metrics
col1, col2, col3 = st.columns(3)
col1.metric("Total Sales", f"${filtered_df['revenue'].sum():,.0f}")
col2.metric("Transactions", f"{len(filtered_df):,}")
col3.metric("Avg Order Value", f"${filtered_df['revenue'].mean():,.2f}")

# Display chart
st.line_chart(filtered_df.set_index('date')['revenue'])

### Deploying Your App

Once you've written your app code:

1. Save it to a file (e.g., `app.py`) in your Databricks workspace or Git repository
2. Use the UI to create a new app and point it to your file
3. Or use the Databricks CLI:

```bash
databricks apps deploy my-app --source-code-path ./app.py
```

📖 **Resource:** [Deploy apps with the CLI](https://docs.databricks.com/en/dev-tools/databricks-apps/deploy-app-cli.html)

## Method 3: Apps with AI Agent Integration

One of the most powerful patterns is combining Databricks Apps with AI agents. This lets you build chatbots and assistants that can interact with your data, answer questions, and take actions.

Below is an example using **Gradio** to create a chatbot interface connected to a deployed AI agent.

In [ ]:
# Example: Gradio chatbot app connected to an AI agent
# This app provides a chat interface for your deployed agent

import gradio as gr
from databricks.agents import get_deployments_client
import os

# Initialize the agent client
client = get_deployments_client()
agent_endpoint = "your_agent_endpoint_name"

def chat_with_agent(message, history):
    """
    Send user message to agent and return response
    """
    response = client.predict(
        endpoint=agent_endpoint,
        inputs={"messages": [{"role": "user", "content": message}]}
    )
    
    return response["choices"][0]["message"]["content"]

# Create Gradio interface
demo = gr.ChatInterface(
    fn=chat_with_agent,
    title="🤖 Customer Support AI Assistant",
    description="Ask questions about our products, orders, and policies",
    examples=[
        "What were our total sales last quarter?",
        "Show me the top 5 selling products",
        "What is the return policy?"
    ]
)

if __name__ == "__main__":
    demo.launch()

## Advanced Patterns

### Multi-Page Applications

Streamlit supports multi-page apps out of the box. Create a `pages/` directory with multiple Python files, and Streamlit will automatically create navigation:

```
app.py                  # Main page
pages/
  1_📊_Analytics.py     # Analytics page
  2_🤖_AI_Chat.py       # AI chat page
  3_⚙️_Settings.py      # Settings page
```

### Authentication and Access Control

Apps automatically inherit Databricks authentication. You can control access using:
* Unity Catalog permissions (table/schema level)
* App-level permissions (who can view/edit the app)
* Custom logic using `dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()`

### Connecting to Unity Catalog

Apps running on Databricks have native access to Unity Catalog. Use the SQL connector or Spark session to query tables securely:

```python
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.table("main.default.my_table")
```

### Real-Time Data Updates

For streaming data or real-time dashboards, use Streamlit's `st.experimental_rerun()` or polling patterns:

```python
import time
placeholder = st.empty()

while True:
    with placeholder.container():
        # Fetch latest data
        df = get_latest_data()
        st.dataframe(df)
    time.sleep(5)  # Refresh every 5 seconds
```

## Example: ML Model Inference App

This example shows how to build an app that serves predictions from a registered MLflow model.

In [ ]:
# Example: Streamlit app for ML model predictions

import streamlit as st
import mlflow
import pandas as pd

st.title("🔮 Customer Churn Prediction")

# Load registered model from Unity Catalog
model_name = "main.default.churn_prediction_model"
model = mlflow.pyfunc.load_model(f"models:/{model_name}/production")

st.write("Enter customer information to predict churn risk:")

# Input fields
col1, col2 = st.columns(2)
with col1:
    tenure = st.number_input("Tenure (months)", min_value=0, max_value=100, value=12)
    monthly_charges = st.number_input("Monthly Charges ($)", min_value=0.0, value=65.0)
    
with col2:
    contract_type = st.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
    support_tickets = st.number_input("Support Tickets (last 30 days)", min_value=0, value=2)

if st.button("Predict Churn Risk"):
    # Create input DataFrame
    input_data = pd.DataFrame({
        'tenure': [tenure],
        'monthly_charges': [monthly_charges],
        'contract_type': [contract_type],
        'support_tickets': [support_tickets]
    })
    
    # Get prediction
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]
    
    # Display result
    if probability > 0.7:
        st.error(f"⚠️ High Churn Risk: {probability*100:.1f}%")
    elif probability > 0.4:
        st.warning(f"⚡ Medium Churn Risk: {probability*100:.1f}%")
    else:
        st.success(f"✅ Low Churn Risk: {probability*100:.1f}%")

## Example: Interactive Data Explorer

A general-purpose app for exploring any Unity Catalog table with filters and visualizations.

In [ ]:
# Example: Universal data exploration app

import streamlit as st
from pyspark.sql import SparkSession
import plotly.express as px

st.title("🔍 Data Explorer")

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Table selector
catalog = st.text_input("Catalog", value="main")
schema = st.text_input("Schema", value="default")
table = st.text_input("Table", value="my_table")

if st.button("Load Data"):
    try:
        # Load table
        full_table_name = f"{catalog}.{schema}.{table}"
        df = spark.table(full_table_name).limit(10000).toPandas()
        
        st.success(f"Loaded {len(df)} rows from {full_table_name}")
        
        # Display summary
        st.subheader("Data Preview")
        st.dataframe(df.head(100))
        
        # Column statistics
        st.subheader("Summary Statistics")
        st.dataframe(df.describe())
        
        # Visualization options
        st.subheader("Visualizations")
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        
        if len(numeric_cols) >= 2:
            col1, col2 = st.columns(2)
            with col1:
                x_col = st.selectbox("X-axis", numeric_cols)
            with col2:
                y_col = st.selectbox("Y-axis", numeric_cols, index=1 if len(numeric_cols) > 1 else 0)
            
            fig = px.scatter(df, x=x_col, y=y_col, title=f"{x_col} vs {y_col}")
            st.plotly_chart(fig)
        
    except Exception as e:
        st.error(f"Error loading table: {str(e)}")

## Comprehensive Resource Library

### 📚 **Official Documentation**
* [Databricks Apps - Complete Guide](https://docs.databricks.com/en/dev-tools/databricks-apps/index.html)
* [Create and Deploy Apps](https://docs.databricks.com/en/dev-tools/databricks-apps/create-app.html)
* [App Configuration and Settings](https://docs.databricks.com/en/dev-tools/databricks-apps/app-config.html)
* [Apps with AI Agents](https://docs.databricks.com/en/generative-ai/agent-framework/index.html)

### 🛠️ **Framework Documentation**
* [Streamlit Documentation](https://docs.streamlit.io/)
* [Gradio Documentation](https://www.gradio.app/docs/)
* [Dash by Plotly](https://dash.plotly.com/)
* [Flask Documentation](https://flask.palletsprojects.com/)

### 🎓 **Hands-On Tutorials**
* [Build Your First Databricks App](https://www.databricks.com/learn/tutorials)
* [Deploying ML Models as Apps](https://docs.databricks.com/en/machine-learning/model-serving/index.html)

### 🏗️ **Architecture Patterns**
* [Production App Best Practices](https://docs.databricks.com/en/dev-tools/databricks-apps/best-practices.html)
* [Security and Governance for Apps](https://docs.databricks.com/en/security/index.html)

### 💡 **Community Resources**
* [Databricks Community Forums](https://community.databricks.com/)
* [Reddit - r/databricks](https://www.reddit.com/r/databricks/)
* [GitHub Examples](https://github.com/databricks)